In [1]:
""" RUN THIS CELL TO GET THE RIGHT FORMATTING """
import requests
from IPython.core.display import HTML
css_file = 'https://raw.githubusercontent.com/bsethwalker/clemson-cs4300/main/css/cpsc6300.css'
styles = requests.get(css_file).text
HTML(styles)

In [3]:
from google.colab import drive
drive.mount('/content/drive')
%cd '/content/drive/MyDrive/Colab-Notebooks/CPSC-6300/Homework-7'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab-Notebooks/CPSC-6300/Homework-7


In [4]:
!ls

 Algerian_forest_fires_dataset_UPDATE-1.csv
 housing.csv
 Lab6_Assignment.ipynb
 Lab_Outlier_Detection__Model_Selection__Cross_Validation.ipynb
 OnlineNewsPopularity.csv
'UGBOKO-Homework 7 Submission.ipynb'


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler

<h2> Feature Selection on OnlineNewsPopularity </h2>

Exercise
1. Load the dataset
2. Drop the Column which isn't required


In [17]:
 # read the csv file from the link provided
 # drop the column that is not required from the dataset(url)
data = pd.read_csv("/content/drive/MyDrive/Colab-Notebooks/CPSC-6300/Homework-7/OnlineNewsPopularity.csv")
data

,url,timedelta,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,num_hrefs,num_self_hrefs,num_imgs,...,min_positive_polarity,max_positive_polarity,avg_negative_polarity,min_negative_polarity,max_negative_polarity,title_subjectivity,title_sentiment_polarity,abs_title_subjectivity,abs_title_sentiment_polarity,shares
0,http://mashable.com/2013/01/07/amazon-instant-...,731.0,12.0,219.0,0.663594,1.0,0.815385,4.0,2.0,1.0,...,0.100000,0.70,-0.350000,-0.600,-0.200000,0.500000,-0.187500,0.000000,0.187500,593
1,http://mashable.com/2013/01/07/ap-samsung-spon...,731.0,9.0,255.0,0.604743,1.0,0.791946,3.0,1.0,1.0,...,0.033333,0.70,-0.118750,-0.125,-0.100000,0.000000,0.000000,0.500000,0.000000,711
2,http://mashable.com/2013/01/07/apple-40-billio...,731.0,9.0,211.0,0.575130,1.0,0.663866,3.0,1.0,1.0,...,0.100000,1.00,-0.466667,-0.800,-0.133333,0.000000,0.000000,0.500000,0.000000,1500
3,http://mashable.com/2013/01/07/astronaut-notre...,731.0,9.0,531.0,0.503788,1.0,0.665635,9.0,0.0,1.0,...,0.136364,0.80,-0.369697,-0.600,-0.166667,0.000000,0.000000,0.500000,0.000000,1200
4,http://mashable.com/2013/01/07/att-u-verse-apps/,731.0,13.0,1072.0,0.415646,1.0,0.540890,19.0,19.0,20.0,...,0.033333,1.00,-0.220192,-0.500,-0.050000,0.454545,0.136364,0.045455,0.136364,505
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39639,http://mashable.com/2014/12/27/samsung-app-aut...,8.0,11.0,346.0,0.529052,1.0,0.684783,9.0,7.0,1.0,...,0.100000,0.75,-0.260000,-0.500,-0.125000,0.100000,0.000000,0.400000,0.000000,1800
39640,http://mashable.com/2014/12/27/seth-rogen-jame...,8.0,12.0,328.0,0.696296,1.0,0.885057,9.0,7.0,3.0,...,0.136364,0.70,-0.211111,-0.400,-0.100000,0.300000,1.000000,0.200000,1.000000,1900
39641,http://mashable.com/2014/12/27/son-pays-off-mo...,8.0,10.0,442.0,0.516355,1.0,0.644128,24.0,1.0,12.0,...,0.136364,0.50,-0.356439,-0.800,-0.166667,0.454545,0.136364,0.045455,0.136364,1900
39642,http://mashable.com/2014/12/27/ukraine-blasts/,8.0,6.0,682.0,0.539493,1.0,0.692661,10.0,1.0,1.0,...,0.062500,0.50,-0.205246,-0.500,-0.012500,0.000000,0.000000,0.500000,0.000000,1100


In [18]:
data = data.drop(data.columns[0], axis=1)
data

,timedelta,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,num_hrefs,num_self_hrefs,num_imgs,num_videos,...,min_positive_polarity,max_positive_polarity,avg_negative_polarity,min_negative_polarity,max_negative_polarity,title_subjectivity,title_sentiment_polarity,abs_title_subjectivity,abs_title_sentiment_polarity,shares
0,731.0,12.0,219.0,0.663594,1.0,0.815385,4.0,2.0,1.0,0.0,...,0.100000,0.70,-0.350000,-0.600,-0.200000,0.500000,-0.187500,0.000000,0.187500,593
1,731.0,9.0,255.0,0.604743,1.0,0.791946,3.0,1.0,1.0,0.0,...,0.033333,0.70,-0.118750,-0.125,-0.100000,0.000000,0.000000,0.500000,0.000000,711
2,731.0,9.0,211.0,0.575130,1.0,0.663866,3.0,1.0,1.0,0.0,...,0.100000,1.00,-0.466667,-0.800,-0.133333,0.000000,0.000000,0.500000,0.000000,1500
3,731.0,9.0,531.0,0.503788,1.0,0.665635,9.0,0.0,1.0,0.0,...,0.136364,0.80,-0.369697,-0.600,-0.166667,0.000000,0.000000,0.500000,0.000000,1200
4,731.0,13.0,1072.0,0.415646,1.0,0.540890,19.0,19.0,20.0,0.0,...,0.033333,1.00,-0.220192,-0.500,-0.050000,0.454545,0.136364,0.045455,0.136364,505
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39639,8.0,11.0,346.0,0.529052,1.0,0.684783,9.0,7.0,1.0,1.0,...,0.100000,0.75,-0.260000,-0.500,-0.125000,0.100000,0.000000,0.400000,0.000000,1800
39640,8.0,12.0,328.0,0.696296,1.0,0.885057,9.0,7.0,3.0,48.0,...,0.136364,0.70,-0.211111,-0.400,-0.100000,0.300000,1.000000,0.200000,1.000000,1900
39641,8.0,10.0,442.0,0.516355,1.0,0.644128,24.0,1.0,12.0,1.0,...,0.136364,0.50,-0.356439,-0.800,-0.166667,0.454545,0.136364,0.045455,0.136364,1900
39642,8.0,6.0,682.0,0.539493,1.0,0.692661,10.0,1.0,1.0,0.0,...,0.062500,0.50,-0.205246,-0.500,-0.012500,0.000000,0.000000,0.500000,0.000000,1100


Exercise
1. Scale the data using a appropriate scaler and re-asign the column names after scaling.
2. The function below should return scaled result in the form of DataFrame

In [19]:
# hint: Use MinMaxScaler for scaling

from sklearn.preprocessing import MinMaxScaler

def scale_data_minmax(df):
    """
    Scale the data using MinMaxScaler and return as DataFrame with column names
    """
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(df)
    scaled_df = pd.DataFrame(scaled_data, columns=df.columns)
    return scaled_df

# Scale your data with MinMaxScaler
scaled_news = scale_data_minmax(data)
scaled_news

,timedelta,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,num_hrefs,num_self_hrefs,num_imgs,num_videos,...,min_positive_polarity,max_positive_polarity,avg_negative_polarity,min_negative_polarity,max_negative_polarity,title_subjectivity,title_sentiment_polarity,abs_title_subjectivity,abs_title_sentiment_polarity,shares
0,1.0,0.476190,0.025844,0.000947,0.00096,0.001254,0.013158,0.017241,0.007812,0.000000,...,0.100000,0.70,0.650000,0.400,0.800000,0.500000,0.406250,0.000000,0.187500,0.000702
1,1.0,0.333333,0.030092,0.000863,0.00096,0.001218,0.009868,0.008621,0.007812,0.000000,...,0.033333,0.70,0.881250,0.875,0.900000,0.000000,0.500000,1.000000,0.000000,0.000842
2,1.0,0.333333,0.024900,0.000820,0.00096,0.001021,0.009868,0.008621,0.007812,0.000000,...,0.100000,1.00,0.533333,0.200,0.866667,0.000000,0.500000,1.000000,0.000000,0.001778
3,1.0,0.333333,0.062662,0.000719,0.00096,0.001024,0.029605,0.000000,0.007812,0.000000,...,0.136364,0.80,0.630303,0.400,0.833333,0.000000,0.500000,1.000000,0.000000,0.001422
4,1.0,0.523810,0.126505,0.000593,0.00096,0.000832,0.062500,0.163793,0.156250,0.000000,...,0.033333,1.00,0.779808,0.500,0.950000,0.454545,0.568182,0.090909,0.136364,0.000598
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39639,0.0,0.428571,0.040831,0.000755,0.00096,0.001054,0.029605,0.060345,0.007812,0.010989,...,0.100000,0.75,0.740000,0.500,0.875000,0.100000,0.500000,0.800000,0.000000,0.002133
39640,0.0,0.476190,0.038707,0.000993,0.00096,0.001362,0.029605,0.060345,0.023438,0.527473,...,0.136364,0.70,0.788889,0.600,0.900000,0.300000,1.000000,0.400000,1.000000,0.002252
39641,0.0,0.380952,0.052160,0.000737,0.00096,0.000991,0.078947,0.008621,0.093750,0.010989,...,0.136364,0.50,0.643561,0.200,0.833333,0.454545,0.568182,0.090909,0.136364,0.002252
39642,0.0,0.190476,0.080481,0.000770,0.00096,0.001066,0.032895,0.008621,0.007812,0.000000,...,0.062500,0.50,0.794754,0.500,0.987500,0.000000,0.500000,1.000000,0.000000,0.001303


5. Perform train_test_split

In [20]:
X = data.loc[:,:' abs_title_sentiment_polarity']
y = data[' shares']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=50)
y_train = pd.DataFrame(y_train)
y_test = pd.DataFrame(y_test)
y_train =y_train.astype('int')

6. Write a function which returns the list of k-Best features where k being the number of features required

In [ ]:
#use chi2


In [24]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2

# Scale data to 0-1 range (non-negative)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)

def select_k_best_features(X, y, k):
    selector = SelectKBest(score_func=chi2, k=k)
    selector.fit(X, y)
    best_features = X.columns[selector.get_support()]
    scores = selector.scores_[selector.get_support()]
    return pd.DataFrame({'Feature': best_features, 'Chi2 Score': scores}).sort_values(by='Chi2 Score', ascending=False)

top_features = select_k_best_features(X_train_scaled, y_train, k=10)
print("\nTop 10 Features based on Chi-Square Test:")
print(top_features)


Top 10 Features based on Chi-Square Test:
                          Feature   Chi2 Score
1   data_channel_is_entertainment  1677.155035
4           data_channel_is_world  1670.875085
9                      is_weekend  1464.377407
2          data_channel_is_socmed  1333.938100
0       data_channel_is_lifestyle  1327.712480
7             weekday_is_saturday  1298.312137
8               weekday_is_sunday  1169.630312
6            weekday_is_wednesday  1162.946294
5               weekday_is_monday  1151.027944
3            data_channel_is_tech  1140.236808


Print the results

<h3>=> Describe about feature selection and explain your code in detail?</h3>


**Feature selection** is the process of identifying and selecting the most informative input variables (features) that contribute the most to predicting the target variable. Its main goals are:



*   Improve model performance: Removing irrelevant or redundant features reduces overfitting and improves generalization.
*   Reduce computational cost: Fewer features mean faster training and testing.
*   Enhance interpretability: Models become easier to understand when only meaningful variables are used.

**Chi-Square Feature Selection**: In this assignment, I used a filter method with the Chi-square (χ²) statistical test to evaluate the dependency between each feature and the target variable. The chi-square test checks whether the distribution of feature values is independent of the target variable; Features with higher chi-square scores are more strongly associated with the output and are therefore more informative; The test requires all input features to be non-negative, hence the need for scaling.

**Step-by-Step Code Explanation**

I loaded the Online News Popularity dataset and removed the 'url' column, which is not useful for modeling. Because the chi-square test requires non-negative inputs, I scaled all features to the range [0, 1] using MinMaxScaler. This preserves the relationships among features while ensuring compatibility with the test. Then I selected the 10 best features and display their chi-square scores. The top-ranked features are those that show the strongest statistical relationship with the number of article shares.

In [ ]:
# Answer


<h2> Model selection on Algerian_forest_fires_dataset_UPDATE-1 dataset  </h2>
<h3># Exercise (Hint use Logistic Regression, SVC, Ridge and Lasso to compare the models.)</h3>
<h4>LinearSVC  Vs  Logistic regression</h4>
<p>Your task is to findout which of the above models is best suited for the given dataset and give reasons in this scenario. </p>
<p>Also, you need to give scenarios which each of these Models work better over the other.</p>


1. Load the dataset

In [25]:
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

data = pd.read_csv("/content/drive/MyDrive/Colab-Notebooks/CPSC-6300/Homework-7/Algerian_forest_fires_dataset_UPDATE-1.csv")
data

,day,month,year,Temperature,RH,Ws,Rain,FFMC,DMC,DC,ISI,BUI,FWI,Classes
0,01,06,2012,29,57,18,0,65.7,3.4,7.6,1.3,3.4,0.5,not fire
1,02,06,2012,29,61,13,1.3,64.4,4.1,7.6,1,3.9,0.4,not fire
2,03,06,2012,26,82,22,13.1,47.1,2.5,7.1,0.3,2.7,0.1,not fire
3,04,06,2012,25,89,13,2.5,28.6,1.3,6.9,0,1.7,0,not fire
4,05,06,2012,27,77,16,0,64.8,3,14.2,1.2,3.9,0.5,not fire
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,26,09,2012,30,65,14,0,85.4,16,44.5,4.5,16.9,6.5,fire
242,27,09,2012,28,87,15,4.4,41.1,6.5,8,0.1,6.2,0,not fire
243,28,09,2012,27,87,29,0.5,45.9,3.5,7.9,0.4,3.4,0.2,not fire
244,29,09,2012,24,54,18,0.1,79.7,4.3,15.2,1.7,5.1,0.7,not fire


In [26]:
df = data.loc[124:]
df = pd.concat([data.loc[:120], data.loc[125:]])
df['Classes  '] = df['Classes  '].str.strip()

In [27]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(df['Classes  '])
df['Classes  '] = le.transform(df['Classes  '])
df = df.drop(index=[167])
df

,day,month,year,Temperature,RH,Ws,Rain,FFMC,DMC,DC,ISI,BUI,FWI,Classes
0,01,06,2012,29,57,18,0,65.7,3.4,7.6,1.3,3.4,0.5,1
1,02,06,2012,29,61,13,1.3,64.4,4.1,7.6,1,3.9,0.4,1
2,03,06,2012,26,82,22,13.1,47.1,2.5,7.1,0.3,2.7,0.1,1
3,04,06,2012,25,89,13,2.5,28.6,1.3,6.9,0,1.7,0,1
4,05,06,2012,27,77,16,0,64.8,3,14.2,1.2,3.9,0.5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,26,09,2012,30,65,14,0,85.4,16,44.5,4.5,16.9,6.5,0
242,27,09,2012,28,87,15,4.4,41.1,6.5,8,0.1,6.2,0,1
243,28,09,2012,27,87,29,0.5,45.9,3.5,7.9,0.4,3.4,0.2,1
244,29,09,2012,24,54,18,0.1,79.7,4.3,15.2,1.7,5.1,0.7,1


* Drop the unnecessary columns and use train_test_split

In [30]:
df = df.drop(columns=['day', 'month', 'year'], errors='ignore')

X = df.drop('Classes  ', axis=1)
y = df['Classes  ']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Apply Logistic Regression, SVC, Ridge and Lasso to compare the models.

In [31]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVC": LinearSVC(max_iter=1000),
    "Ridge Classifier": RidgeClassifier()
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

print("\nModel Comparison Results:")
for model, score in results.items():
    print(f"{model}: {score:.4f}")


Model Comparison Results:
Logistic Regression: 0.9726
Linear SVC: 0.9726
Ridge Classifier: 0.9589
